# Factor Weight Regression v2 — 700 Small/Mid-Cap Stocks

Linear-regression (OLS) alternative to `optimization_ranking.ipynb`. Instead of
searching for weights that maximize Spearman rank correlation, this fits

    return ~ w1*accel + w2*momentum + w3*growth + ...

directly via `np.linalg.lstsq` (closed form). The whole CV runs in **seconds**
instead of minutes, and the coefficients are directly interpretable.

**Same as the optimizer notebook:**
1. **Multi-period**: 12, 24, 36 month lookbacks (~1800 observations)
2. **Reduced factors**: 7 instead of 9 (merged upside+conviction, kept long_term)
3. **Sector-neutralized**: subtract sector median return to find stock-level signals

**Different:**
- **OLS fit** instead of scipy minimize + coordinate descent (~10x faster)
- **Winsorizing is a toggle** — flip `WINSORIZE` below to compare capped vs raw returns

**Pipeline:**
1. Fetch factor scores for ~700 stocks across 3 lookback periods
2. Preprocess: (optionally) winsorize returns, neutralize by sector
3. 80/20 CV (5 splits) fitting OLS, reporting Spearman/Pearson out-of-sample
4. Per-strategy fit + quintile analysis
5. Copy-paste block for ranking.py

In [ ]:
import os, sys

REPO_ROOT = os.path.dirname(os.path.abspath('__file__'))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from portfolio.smid_data_fetcher import fetch_all, load_csv, FACTOR_NAMES, ADJUSTMENT_NAMES
from portfolio.smid_regression import (
    cross_validate, strategy_cv, fit_regression,
    REDUCED_FACTORS, FULL_FACTORS,
)
from portfolio.smid_optimizer import (
    format_weights_table, quintile_analysis,
    score_records, round_weights, pearson, spearman,
    preprocess, _compute_reduced_factors,
)
import numpy as np

print("Setup complete.")
print(f"Full factors (9): {FULL_FACTORS}")
print(f"Reduced factors (7): {REDUCED_FACTORS}")

## Toggle: Winsorize ON/OFF

This is the one knob that differs from the optimizer notebook.

- `WINSORIZE = True`  → cap returns at `+/- WINSORIZE_CAP`% before fitting (robust)
- `WINSORIZE = False` → fit on raw returns (a few +2000% outliers will dominate)

Every CV cell below reads these two variables, so flip them here and re-run.

In [ ]:
WINSORIZE = True      # set False to fit on raw (uncapped) returns
WINSORIZE_CAP = 200.0 # only used when WINSORIZE is True

print(f"Winsorize: {'ON  (+/-%.0f%%)' % WINSORIZE_CAP if WINSORIZE else 'OFF (raw returns)'}")

## Step 1: Fetch Data (3 lookback periods)

Fetches factor scores for ~700 tickers at 12, 24, and 36 month lookbacks.
First run takes ~2-3 hours. Results cached to `output/smid_optimization_data.csv`.
Supports resume — if interrupted, re-run and it picks up where it left off.

In [ ]:
# Set force_refresh=True to refetch everything from scratch
# Quick test: set MAX_TICKERS=40 for ~5 min run, None for full ~700 (~3 hours)
MAX_TICKERS = 40  # Set to None for full run

records = fetch_all(periods=[12, 24, 36], force_refresh=False, max_tickers=MAX_TICKERS)
print(f"\nTotal records: {len(records)}")

# Period distribution
periods = {}
for r in records:
    pm = r["period_months"]
    periods[pm] = periods.get(pm, 0) + 1
print(f"Per period: {periods}")

# Strategy distribution
strats = {}
for r in records:
    s = r["strategy"]
    strats[s] = strats.get(s, 0) + 1
print(f"Strategy distribution: {strats}")

# Return distribution (raw, before winsorizing)
returns = [r["actual_return"] for r in records]
print(f"\nReturn stats (raw):")
print(f"  Mean:   {np.mean(returns):+.1f}%")
print(f"  Median: {np.median(returns):+.1f}%")
print(f"  Std:    {np.std(returns):.1f}%")
print(f"  Min:    {np.min(returns):+.1f}%")
print(f"  Max:    {np.max(returns):+.1f}%")
print(f"  >200%:  {sum(1 for r in returns if r > 200)} stocks")
print(f"  <-90%:  {sum(1 for r in returns if r < -90)} stocks")

## Step 2: Individual Factor Correlations

Raw correlations before fitting. Shows which factors have
predictive power on their own.

In [ ]:
print("=== RAW (all 9 factors) ===")
print(f"{'Factor':<22} {'Pearson':>8} {'Spearman':>9}")
print("-" * 41)
for f in FACTOR_NAMES:
    fv = [r[f] for r in records]
    rv = [r["actual_return"] for r in records]
    print(f"{f:<22} {pearson(fv, rv):+.4f}   {spearman(fv, rv):+.4f}")

print(f"\n=== REDUCED (7 factors, merged analyst_sentiment) ===")
print(f"{'Factor':<22} {'Pearson':>8} {'Spearman':>9}")
print("-" * 41)
for f in REDUCED_FACTORS:
    fv = [_compute_reduced_factors(r)[f] for r in records]
    rv = [r["actual_return"] for r in records]
    print(f"{f:<22} {pearson(fv, rv):+.4f}   {spearman(fv, rv):+.4f}")

print(f"\n=== ADJUSTMENTS ===")
for a in ADJUSTMENT_NAMES:
    av = [r[a] for r in records]
    rv = [r["actual_return"] for r in records]
    print(f"{a:<22} {pearson(av, rv):+.4f}   {spearman(av, rv):+.4f}")

## Step 3: 80/20 Cross-Validation (All Strategies Pooled)

Fits 7 factor weights via OLS regression.
Preprocessing: (optionally) winsorize returns, sector-neutralize.
5 random 80/20 splits, weights averaged across splits.

Uses the `WINSORIZE` / `WINSORIZE_CAP` toggle from the top of the notebook.

In [ ]:
cv_result = cross_validate(
    records, n_splits=5, test_size=0.2,
    metric="spearman", seed=42,
    use_reduced=True,
    winsorize=WINSORIZE,
    winsorize_cap=WINSORIZE_CAP,
    sector_neutral=True,
)

metric = cv_result["metric"]
print(f"\n{'='*60}")
print(f"SUMMARY (regression, all strategies pooled)")
print(f"{'='*60}")
print(f"Winsorize:           {'ON (+/-%.0f%%)' % WINSORIZE_CAP if WINSORIZE else 'OFF (raw)'}")
print(f"Mean test {metric}:  {cv_result[f'mean_test_{metric}']:+.4f} (+/-{cv_result[f'std_test_{metric}']:.4f})")
print(f"Mean train {metric}: {cv_result[f'mean_train_{metric}']:+.4f}")
print(f"Mean train R^2:      {cv_result['mean_train_r_squared']:+.4f}")
print(f"Overfit gap:         {cv_result['overfit_gap']:+.4f}")
print(f"Adj multiplier:      {cv_result['avg_adj_multiplier']:.2f}")
print(f"\nOptimal weights (7 reduced factors):")
for f in REDUCED_FACTORS:
    print(f"  {f:<22} {cv_result['avg_weights'][f]*100:>5.1f}%")

## Step 3b (optional): Compare Winsorize ON vs OFF

Runs the CV both ways in one shot so you can see how much the
return outliers move the weights. Does not depend on the toggle.

In [ ]:
res_on = cross_validate(records, n_splits=5, metric="spearman", seed=42,
                        use_reduced=True, winsorize=True, winsorize_cap=200.0,
                        sector_neutral=True)
res_off = cross_validate(records, n_splits=5, metric="spearman", seed=42,
                         use_reduced=True, winsorize=False, sector_neutral=True)

print(f"\n{'='*60}")
print("WINSORIZE ON vs OFF")
print(f"{'='*60}")
print(f"{'':<22} {'ON (+/-200%)':>14} {'OFF (raw)':>14}")
print("-" * 52)
print(f"{'Mean test spearman':<22} {res_on['mean_test_spearman']:>+14.4f} {res_off['mean_test_spearman']:>+14.4f}")
print(f"{'Mean train R^2':<22} {res_on['mean_train_r_squared']:>+14.4f} {res_off['mean_train_r_squared']:>+14.4f}")
print(f"\n{'Factor':<22} {'ON':>14} {'OFF':>14}")
print("-" * 52)
for f in REDUCED_FACTORS:
    print(f"{f:<22} {res_on['avg_weights'][f]*100:>13.1f}% {res_off['avg_weights'][f]*100:>13.1f}%")

## Step 4: Quintile Analysis

Score all stocks with fitted weights, split into quintiles,
compare average returns. Uses RAW (non-preprocessed) returns
so the numbers reflect actual portfolio performance.

In [ ]:
print("=== QUINTILE ANALYSIS (raw returns, fitted weights) ===")
quintiles = quintile_analysis(
    records,
    cv_result["avg_weights"],
    adj_mult=cv_result["avg_adj_multiplier"],
    use_reduced=True,
)

## Step 5: Per-Strategy Regression

Separate OLS fit + CV for each strategy. Only meaningful for strategies
with enough observations (>100 stocks per period).

In [ ]:
strat_results = strategy_cv(
    records, n_splits=5, metric="spearman",
    use_reduced=True, winsorize=WINSORIZE,
    winsorize_cap=WINSORIZE_CAP, sector_neutral=True,
)

print(format_weights_table(cv_result, strat_results))

In [ ]:
print("\n=== PER-STRATEGY QUINTILE ANALYSIS ===")
for strat in ["hold_forever", "cycle", "catalyst"]:
    sr = strat_results.get(strat)
    if sr is None:
        continue
    strat_recs = [r for r in records if r["strategy"] == strat]
    print(f"\n--- {strat} ({len(strat_recs)} stocks) ---")
    quintile_analysis(
        strat_recs, sr["avg_weights"],
        adj_mult=sr["avg_adj_multiplier"],
        use_reduced=True,
    )

## Step 6: Current vs Regression Weights

Maps the 7 reduced factors back to the original 9 for comparison.
analyst_sentiment weight is split equally between upside and conviction.

In [ ]:
from portfolio.ranking import STRATEGY_WEIGHTS

strategies = ["hold_forever", "cycle", "catalyst"]

# Map reduced weights back to original 9 factors
def expand_weights(reduced_w):
    """Convert 7 reduced factors back to 9 original factors."""
    sentiment = reduced_w.get("analyst_sentiment", 0)
    return {
        "upside": sentiment / 2,
        "growth": reduced_w.get("growth", 0),
        "accel": reduced_w.get("accel", 0),
        "valuation": reduced_w.get("valuation", 0),
        "long_term": reduced_w.get("long_term", 0),
        "cash_runway": 0.0,  # dropped
        "conviction": sentiment / 2,
        "entry": reduced_w.get("entry", 0),
        "momentum": reduced_w.get("momentum", 0),
    }

factor_labels = {
    "growth": "Growth", "upside": "Upside", "accel": "Acceleration",
    "valuation": "Valuation (P/S)", "long_term": "LT Health",
    "cash_runway": "Cash Runway", "conviction": "Conviction",
    "entry": "Entry", "momentum": "Momentum",
}

print(f"\n{'='*70}")
print(f"CURRENT vs REGRESSION WEIGHTS (mapped to original 9 factors)")
print(f"{'='*70}")

print(f"\n{'Factor':<18}", end="")
for s in strategies:
    print(f" {'Current':>8} {'Regr':>8}", end="")
print()
print("-" * 72)

for f in FULL_FACTORS:
    label = factor_labels.get(f, f)
    print(f"{label:<18}", end="")
    for s in strategies:
        current = STRATEGY_WEIGHTS.get(s, {}).get(f, 0)
        sr = strat_results.get(s)
        if sr:
            expanded = expand_weights(sr["avg_weights"])
        else:
            expanded = expand_weights(cv_result["avg_weights"])
        regr = expanded.get(f, 0)
        print(f" {current*100:>7.1f}% {regr*100:>7.1f}%", end="")
    print()

print(f"\n{'Test Spearman':<18}", end="")
for s in strategies:
    sr = strat_results.get(s)
    if sr:
        print(f" {'':>8} {sr['mean_test_spearman']:>+7.4f}", end="")
    else:
        print(f" {'':>8} {'N/A':>8}", end="")
print()
print(f"\nOverall pooled test Spearman: {cv_result[f'mean_test_{cv_result["metric"]}']:+.4f}")

## Step 7: Copy-Paste Block for ranking.py

In [ ]:
print("STRATEGY_WEIGHTS = {")
for strat in strategies:
    sr = strat_results.get(strat)
    if sr:
        expanded = expand_weights(sr["avg_weights"])
    else:
        expanded = expand_weights(cv_result["avg_weights"])
    print(f'    "{strat}": {{')
    for i, f in enumerate(FULL_FACTORS):
        comma = ","
        print(f'        "{f}":{" " * (14 - len(f))}{expanded[f]:.2f}{comma}')
    print("    },")
print("}")

## Visualization

In [ ]:
try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    x = np.arange(len(FULL_FACTORS))
    width = 0.35
    labels = [factor_labels.get(f, f) for f in FULL_FACTORS]

    for idx, strat in enumerate(strategies):
        ax = axes[idx]
        current_vals = [STRATEGY_WEIGHTS.get(strat, {}).get(f, 0) * 100 for f in FULL_FACTORS]
        sr = strat_results.get(strat)
        if sr:
            expanded = expand_weights(sr["avg_weights"])
            test_metric = sr[f"mean_test_{cv_result['metric']}"]
        else:
            expanded = expand_weights(cv_result["avg_weights"])
            test_metric = cv_result[f"mean_test_{cv_result['metric']}"]
        regr_vals = [expanded.get(f, 0) * 100 for f in FULL_FACTORS]

        ax.barh(x - width/2, current_vals, width, label="Current", alpha=0.7)
        ax.barh(x + width/2, regr_vals, width, label="Regression", alpha=0.7)
        ax.set_yticks(x)
        ax.set_yticklabels(labels)
        ax.set_xlabel("Weight %")
        ax.set_title(f"{strat}\nTest Spearman: {test_metric:+.4f}")
        ax.legend()
        ax.invert_yaxis()

    plt.tight_layout()
    plt.savefig("output/weight_comparison_regression.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved: output/weight_comparison_regression.png")
except ImportError:
    print("matplotlib not installed, skipping visualization")

## Step 8: Save & Push CSV to GitHub

In [ ]:
import subprocess, os

csv_path = "output/smid_optimization_data.csv"
if os.path.exists(csv_path):
    subprocess.run(["git", "add", csv_path], cwd=REPO_ROOT)
    result = subprocess.run(
        ["git", "commit", "-m", "Update optimization data CSV"],
        capture_output=True, text=True, cwd=REPO_ROOT,
    )
    print(result.stdout.strip() or result.stderr.strip())

    result = subprocess.run(
        ["git", "push", "origin", "main"],
        capture_output=True, text=True, cwd=REPO_ROOT,
    )
    if result.returncode == 0:
        print("Pushed CSV to GitHub successfully.")
    else:
        print(f"Push failed: {result.stderr.strip()}")
else:
    print(f"CSV not found at {csv_path}")